# 05 モデル評価インタラクティブノートブック

このノートブックは LightGBM LambdaRank モデルの評価を対話的に確認するためのものです。

## 目次
1. [セットアップ](#1-セットアップ)
2. [モデル読み込み](#2-モデル読み込み)
3. [データ取得と分割](#3-データ取得と分割)
4. [評価指標の計算](#4-評価指標の計算)
5. [特徴量重要度の可視化](#5-特徴量重要度の可視化)
6. [月次評価指標の推移](#6-月次評価指標の推移)
7. [目標指標との比較](#7-目標指標との比較)
8. [レポート生成](#8-レポート生成)

## 1. セットアップ

In [ ]:
import sys
import os
from pathlib import Path

# プロジェクトルートをパスに追加
project_root = Path("../").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import roc_auc_score

# 日本語フォントの設定（環境に応じて変更してください）
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 6)

print("Setup complete.")

In [ ]:
# ============================================================
# 設定: 環境に合わせて変更してください
# ============================================================

# モデルパス
MODEL_PATH = "../src/models/lgbm_ranker_20260217.txt"

# データソース設定
# BigQueryを使う場合: USE_BQ = True, LOCAL_DATA_PATH = None
# ローカルCSVを使う場合: USE_BQ = False, LOCAL_DATA_PATH = "path/to/data.csv"
USE_BQ = False
LOCAL_DATA_PATH = None  # ローカルCSVパス（USE_BQ=Falseの場合に設定）
GCP_PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "your-project-id")

# 分割設定
VALIDATION_MONTHS = 6
TEST_MONTHS = 0  # 0=テスト期間なし

# 除外カラム・カテゴリカルカラム
EXCLUDE_COLUMNS = [
    "race_id", "horse_id", "race_date", "target_place",
    "finish_position", "venue_code", "jockey_id", "trainer_id",
    "created_at", "race_number",
]
CATEGORICAL_COLUMNS = ["course_type", "track_condition"]

print(f"Model: {MODEL_PATH}")
print(f"Data source: {'BigQuery' if USE_BQ else 'Local CSV'}")

## 2. モデル読み込み

In [ ]:
model_path = Path(MODEL_PATH)
booster = lgb.Booster(model_file=str(model_path))

# メタデータ読み込み
meta_path = model_path.with_suffix(".meta.json")
meta = {}
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    print(f"Best iteration: {meta.get('best_iteration')}")
    print(f"Params: {meta.get('params')}")
else:
    print("メタデータファイルが見つかりません")

print(f"\n特徴量数: {len(booster.feature_name())}")

## 3. データ取得と分割

In [ ]:
if USE_BQ:
    from google.cloud import bigquery
    client = bigquery.Client(project=GCP_PROJECT_ID)
    query = f"""
    SELECT *
    FROM `{GCP_PROJECT_ID}.features.training_data`
    ORDER BY race_date, race_id, horse_number
    """
    print("BigQueryからデータを取得中...")
    df = client.query(query).to_dataframe()
elif LOCAL_DATA_PATH:
    print(f"CSVからデータを読み込み中: {LOCAL_DATA_PATH}")
    df = pd.read_csv(LOCAL_DATA_PATH)
else:
    # デモ用のサンプルデータを生成
    print("デモ用サンプルデータを生成します")
    print("実際の使用時は USE_BQ=True または LOCAL_DATA_PATH を設定してください")
    df = None

if df is not None:
    df["race_date"] = pd.to_datetime(df["race_date"])
    df = df.sort_values(["race_date", "race_id", "horse_number"])
    print(f"データ取得完了: {len(df):,} rows, {len(df.columns)} columns")
    print(f"期間: {df['race_date'].min().date()} 〜 {df['race_date'].max().date()}")
    display(df.head(3))

In [ ]:
if df is not None:
    max_date = df["race_date"].max()

    if TEST_MONTHS > 0:
        test_start = max_date - pd.DateOffset(months=TEST_MONTHS) + pd.Timedelta(days=1)
        valid_end = test_start - pd.Timedelta(days=1)
        valid_start = valid_end - pd.DateOffset(months=VALIDATION_MONTHS) + pd.Timedelta(days=1)
        test_df = df[df["race_date"] >= test_start]
        valid_df = df[(df["race_date"] >= valid_start) & (df["race_date"] <= valid_end)]
        train_df = df[df["race_date"] < valid_start]
    else:
        valid_start = max_date - pd.DateOffset(months=VALIDATION_MONTHS) + pd.Timedelta(days=1)
        valid_df = df[df["race_date"] >= valid_start]
        train_df = df[df["race_date"] < valid_start]
        test_df = None

    print(f"Train: {len(train_df):,} rows ({train_df['race_date'].min().date()} 〜 {train_df['race_date'].max().date()})")
    print(f"Valid: {len(valid_df):,} rows ({valid_df['race_date'].min().date()} 〜 {valid_df['race_date'].max().date()})")
    if test_df is not None:
        print(f"Test:  {len(test_df):,} rows ({test_df['race_date'].min().date()} 〜 {test_df['race_date'].max().date()})")

## 4. 評価指標の計算

In [ ]:
def prepare_X(data, exclude_cols, cat_cols):
    """特徴量行列を準備する"""
    feature_cols = [c for c in data.columns if c not in exclude_cols]
    X = data[feature_cols].copy()
    for col in cat_cols:
        if col in X.columns:
            X[col] = X[col].astype("category")
    for col in X.select_dtypes(include="object").columns:
        X[col] = X[col].astype("category")
    return X


def compute_metrics(data, booster, exclude_cols, cat_cols):
    """評価指標を計算する"""
    X = prepare_X(data, exclude_cols, cat_cols)
    positions = data["finish_position"].values.astype(int)
    y_binary = np.where((positions >= 1) & (positions <= 3), 1, 0)
    groups = data.groupby("race_id", sort=False).size().tolist()
    y_pred = booster.predict(X)

    ndcg_list, recall_list = [], []
    start = 0
    for g in groups:
        end = start + g
        true_pos = positions[start:end]
        pred = y_pred[start:end]

        top3_pred_idx = set(np.argsort(pred)[::-1][:3])
        top3_true_idx = set(np.where(true_pos <= 3)[0])
        if top3_true_idx:
            recall_list.append(len(top3_pred_idx & top3_true_idx) / len(top3_true_idx))

        relevance = np.where(true_pos <= 3, 1.0, 0.0)
        pred_order = np.argsort(pred)[::-1]
        dcg = sum(relevance[pred_order[i]] / np.log2(i + 2) for i in range(min(3, g)))
        ideal_order = np.argsort(relevance)[::-1]
        idcg = sum(relevance[ideal_order[i]] / np.log2(i + 2) for i in range(min(3, g)))
        if idcg > 0:
            ndcg_list.append(dcg / idcg)
        start = end

    auc = float(roc_auc_score(y_binary, y_pred)) if len(np.unique(y_binary)) >= 2 else 0.0

    return {
        "ndcg@3": np.mean(ndcg_list) if ndcg_list else 0.0,
        "recall@3": np.mean(recall_list) if recall_list else 0.0,
        "auc": auc,
        "num_races": len(groups),
        "num_rows": len(data),
    }

In [ ]:
if df is not None:
    print("評価指標を計算中...")
    train_metrics = compute_metrics(train_df, booster, EXCLUDE_COLUMNS, CATEGORICAL_COLUMNS)
    valid_metrics = compute_metrics(valid_df, booster, EXCLUDE_COLUMNS, CATEGORICAL_COLUMNS)
    test_metrics = compute_metrics(test_df, booster, EXCLUDE_COLUMNS, CATEGORICAL_COLUMNS) if test_df is not None else None

    results = {
        "Train": train_metrics,
        "Valid": valid_metrics,
    }
    if test_metrics:
        results["Test"] = test_metrics

    metrics_display = pd.DataFrame(results).T
    display(metrics_display.style.format({
        "ndcg@3": "{:.4f}",
        "recall@3": "{:.4f}",
        "auc": "{:.4f}",
        "num_races": "{:,.0f}",
        "num_rows": "{:,.0f}",
    }))

## 5. 特徴量重要度の可視化

In [ ]:
TOP_N = 20

importance = booster.feature_importance(importance_type="gain")
names = booster.feature_name()

df_imp = pd.DataFrame({"feature": names, "importance": importance})
df_imp = df_imp.sort_values("importance", ascending=False).head(TOP_N)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(df_imp["feature"][::-1], df_imp["importance"][::-1], color="#4C72B0")
ax.set_xlabel("Importance (Gain)", fontsize=12)
ax.set_title(f"Feature Importance TOP {TOP_N}", fontsize=14)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

print("\nTOP 20 特徴量重要度:")
display(df_imp.reset_index(drop=True))

## 6. 月次評価指標の推移

In [ ]:
if df is not None:
    df_monthly = df.copy()
    df_monthly["year_month"] = df_monthly["race_date"].dt.to_period("M")
    months = sorted(df_monthly["year_month"].unique())

    monthly_records = []
    for month in months:
        monthly = df_monthly[df_monthly["year_month"] == month]
        if len(monthly) < 10:
            continue
        try:
            m = compute_metrics(monthly, booster, EXCLUDE_COLUMNS, CATEGORICAL_COLUMNS)
            monthly_records.append({"month": str(month), **m})
        except Exception as e:
            print(f"Skipped {month}: {e}")

    if monthly_records:
        metrics_ts = pd.DataFrame(monthly_records)
        metrics_ts["month"] = pd.to_datetime(metrics_ts["month"].astype(str))

        fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
        for ax, col, color, label in zip(
            axes,
            ["ndcg@3", "recall@3", "auc"],
            ["#4C72B0", "#DD8452", "#55A868"],
            ["NDCG@3", "Recall@3", "AUC"],
        ):
            ax.plot(metrics_ts["month"], metrics_ts[col], marker="o", color=color, linewidth=2)
            ax.set_ylabel(label, fontsize=11)
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 1.05)

        axes[-1].set_xlabel("Month", fontsize=11)
        axes[0].set_title("Monthly Evaluation Metrics", fontsize=13)
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()
    else:
        print("月次データが不足しています")

## 7. 目標指標との比較

In [ ]:
if df is not None:
    TARGET = {"ndcg@3": 0.70, "recall@3": 0.80, "auc": 0.70}

    rows = []
    for metric, target in TARGET.items():
        actual = valid_metrics[metric]
        rows.append({
            "指標": metric,
            "目標値": target,
            "実績（検証）": round(actual, 4),
            "達成": "✓" if actual >= target else "✗",
        })

    display(pd.DataFrame(rows).set_index("指標"))

## 8. レポート生成

以下のセルを実行することで `docs/model_evaluation_report.md` が生成されます。

In [ ]:
# レポート生成コマンドの例（ターミナルで実行）
print("以下のコマンドでレポートを生成できます:")
print()
print("# ローカルモデル + BigQuery:")
print(f"python scripts/generate_evaluation_report.py --model-path {MODEL_PATH} --project-id {GCP_PROJECT_ID}")
print()
print("# ローカルモデル + ローカルCSV:")
print(f"python scripts/generate_evaluation_report.py --model-path {MODEL_PATH} --local-data-path /path/to/training_data.csv")

In [ ]:
# このノートブックからスクリプトを直接実行する場合
# (USE_BQ=True またはLOCAL_DATA_PATHが設定されている必要があります)

if df is not None:
    import subprocess
    cmd = [
        "python", "../scripts/generate_evaluation_report.py",
        "--model-path", MODEL_PATH,
        "--validation-months", str(VALIDATION_MONTHS),
    ]
    if USE_BQ:
        cmd.extend(["--project-id", GCP_PROJECT_ID])
    elif LOCAL_DATA_PATH:
        cmd.extend(["--local-data-path", LOCAL_DATA_PATH])

    print("実行コマンド:", " ".join(cmd))
    # result = subprocess.run(cmd, capture_output=True, text=True, cwd=project_root)
    # print(result.stdout)
    # if result.returncode != 0:
    #     print("STDERR:", result.stderr)
    print("(実際に実行するにはコメントアウトを解除してください)")